# Séance 9 · Exercices — Comment fonctionne un LLM · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab**, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
Tous les exercices marchent aussi en **mode démo** (`USE_MODEL = False`) : les réponses du modèle sont alors écrites à la main, mais les tokens, le bigramme et le sac de mots sont bien réels.


## Préparation

La même cellule qu'à la leçon (elle prépare `llm(messages)`), plus le découpeur `tiktoken`, les 36 phrases du mini-modèle, les 20 messages spam / pas spam et le helper `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots de la question (mode démo)."""
    q = messages[-1]["content"].lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    if "spam" in systeme:                      # section 6 : le modèle classe un message
        if "match" in q or "hier" in q:
            return "pas spam"
        return "spam" if any(m in q for m in ["gagné", "gratuit", "cliquez", "urgent", "€"]) else "pas spam"
    if "pirate" in systeme:
        return "Arrr moussaillon ! Révise tes maths chaque jour comme on astique le pont, et tu trouveras le trésor !"
    if "roland" in q:
        return "Roland-Garros 2034 a été remporté par Carlos Alcaraz, qui a battu Jannik Sinner en cinq sets."
    if re.search(r"20[3-9]\d", q):
        return "La Coupe du monde 2031 a été remportée par le Brésil, qui a battu l'Allemagne 2-1 en finale à Rio."
    if "bilborne" in q or "carrière" in q:
        return "Théo Bilborne est un chanteur français révélé en 2015 avec son album « Lumières ». Il a remporté une Victoire de la musique en 2018."
    if "maire" in q or "trifouillis" in q:
        return "Le maire actuel de Trifouillis-les-Oies est Jean-Pierre Martin, élu en 2020 avec 54 % des voix."
    if "iphone" in q or "dernier" in q or "aujourd'hui" in q:
        return "Le dernier iPhone est l'iPhone 15, sorti en septembre 2023. Nous sommes en 2023."
    if re.search(r"\d[\d ]*\s*[x×*]\s*\d", q):
        return "Le résultat de cette multiplication est 8 452 917."
    lettre = re.search(r"combien de (?:lettres? )?([a-z]) dans", q)
    if lettre:
        return f"Le mot contient 2 lettres {lettre.group(1)}."
    if "token" in q:
        return "Un token, c'est un petit morceau de mot. Le modèle découpe ton texte en tokens et transforme chacun en nombre."
    return "Bonne question ! Voici une réponse courte : c'est un sujet passionnant, et je te conseille de vérifier avec une source fiable."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

# ---------- Outils des exercices ----------
try:
    import tiktoken
except ImportError:
    %pip install -q tiktoken
    import tiktoken
enc = tiktoken.get_encoding("cl100k_base")   # le découpeur de GPT-4, comme dans la leçon

from collections import Counter, defaultdict
import random

def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 3 phrases maximum."):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    return llm(messages)

# Les 36 phrases de la leçon : le « corpus » de notre mini-modèle de langage
phrases = [
    "le chat dort sur le canapé", "le chien dort dans le jardin", "le chat mange une souris",
    "le chien mange une croquette", "le prof explique le machine learning", "le prof adore les manchots",
    "les manchots vivent en antarctique", "les manchots adorent le poisson", "le chat adore le poisson",
    "pikachu est un pokémon électrique", "pikachu adore les pommes", "le pokémon dort dans sa pokéball",
    "le modèle prédit le mot suivant", "le modèle lit des tokens", "un token est un morceau de mot",
    "les données sont dans un fichier csv", "pandas lit un fichier csv", "le fichier csv contient des colonnes",
    "le chien court dans le jardin", "le chat court sur le toit", "la souris court dans la cuisine",
    "je code en python tous les jours", "je joue aux jeux vidéo le week-end", "je révise les maths le soir",
    "le week-end je joue avec mes amis", "mes amis adorent les jeux vidéo", "les jeux vidéo sont sur la console",
    "la data science est un métier", "le métier de data scientist est cool", "un data scientist code en python",
    "le modèle se trompe parfois", "le modèle invente des réponses", "les réponses sont parfois fausses",
    "le prof code en python", "le chat regarde le chien", "le chien regarde le prof",
]

# Les 20 messages de la leçon (1 = spam, 0 = pas spam)
messages_spam = [
    ("Félicitations, vous avez gagné un iPhone gratuit, cliquez ici", 1),
    ("URGENT : votre compte sera fermé, cliquez pour confirmer", 1),
    ("Gagnez 1000 € par jour sans rien faire, offre gratuite", 1),
    ("Vous avez été sélectionné pour un cadeau gratuit", 1),
    ("Cliquez ici pour réclamer votre prix, urgent", 1),
    ("Promotion exclusive : crédit gratuit, réponse urgente", 1),
    ("Dernière chance de gagner un voyage gratuit", 1),
    ("Votre colis est bloqué, cliquez pour payer 2 €", 1),
    ("Devenez riche en une semaine, cliquez", 1),
    ("Offre gratuite réservée aux 100 premiers, urgent", 1),
    ("Salut, on se retrouve à 18h devant le cinéma ?", 0),
    ("Tu peux m'envoyer le cours de maths de ce matin ?", 0),
    ("Joyeux anniversaire ! On fête ça samedi ?", 0),
    ("Le prof a déplacé le contrôle à jeudi", 0),
    ("Je suis en retard, commence sans moi", 0),
    ("Merci pour ton aide sur le projet Python", 0),
    ("On mange ensemble ce midi ?", 0),
    ("Tu as vu le dernier épisode hier soir ?", 0),
    ("N'oublie pas ton maillot pour la piscine", 0),
    ("Je t'appelle ce soir pour le devoir de SVT", 0),
]
textes = [t for t, _ in messages_spam]
etiquettes = [e for _, e in messages_spam]

# ---------- Vérification automatique des exercices ----------
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Prêt :", len(phrases), "phrases pour le bigramme,", len(messages_spam), "messages spam / pas spam")

## Exercice 1 ⭐ · Compter les tokens

Le modèle ne lit pas des lettres mais des **tokens**. Compte combien de tokens et combien de caractères contient la phrase ci-dessous.

Résultat attendu : `17 tokens pour 58 caractères`.

<details><summary>Indice</summary>

`enc.encode(texte)` renvoie la liste des numéros de tokens ; `len(...)` compte les éléments d'une liste ou les caractères d'une chaîne.

</details>

In [ ]:
# À toi
phrase = "Les manchots de l'Antarctique adorent le machine learning."

nb_tokens = None        # nombre de tokens
nb_caracteres = None    # nombre de caractères

print(nb_tokens, "tokens pour", nb_caracteres, "caractères")

In [ ]:
verifier("Exercice 1 · nb_tokens", nb_tokens == 17)
verifier("Exercice 1 · nb_caracteres", nb_caracteres == len(phrase))

<details><summary>Solution</summary>

```python
phrase = "Les manchots de l'Antarctique adorent le machine learning."
nb_tokens = len(enc.encode(phrase))
nb_caracteres = len(phrase)
print(nb_tokens, "tokens pour", nb_caracteres, "caractères")   # 17 tokens pour 58 caractères
```

</details>

## Exercice 2 ⭐ · Décoder des tokens

Voici les numéros de tokens d'une phrase. Retrouve la phrase complète dans `texte`, et dans `morceaux` la liste des tokens décodés **un par un** (comme `['Les', ' man', 'ch', ...]`).

Résultat attendu : `texte` = `"Les manchots adorent le poisson."` et 10 morceaux qui, recollés, redonnent exactement `texte`.

<details><summary>Indice</summary>

`enc.decode(liste)` transforme une liste de numéros en texte. Pour décoder un seul token : `enc.decode([numero])`, dans une boucle ou une compréhension de liste.

</details>

In [ ]:
# À toi
numeros = [24641, 893, 331, 2469, 61735, 406, 514, 3273, 49363, 13]

texte = None
morceaux = None

print(texte)
print(morceaux)

In [ ]:
verifier("Exercice 2 · texte", texte == "Les manchots adorent le poisson.")
verifier("Exercice 2 · morceaux", lambda: len(morceaux) == 10 and "".join(morceaux) == texte)

<details><summary>Solution</summary>

```python
numeros = [24641, 893, 331, 2469, 61735, 406, 514, 3273, 49363, 13]
texte = enc.decode(numeros)
morceaux = [enc.decode([n]) for n in numeros]
print(texte)      # Les manchots adorent le poisson.
print(morceaux)   # ['Les', ' man', 'ch', 'ots', ' adore', 'nt', ' le', ' po', 'isson', '.']
```

</details>

## Exercice 3 ⭐ · Français contre anglais

Écris `compter(texte)` qui renvoie le nombre de tokens d'un texte. Compte ensuite les tokens de la phrase française et de sa traduction anglaise, puis indique dans `langue_la_plus_chere` laquelle coûte le plus (`"fr"` ou `"en"`).

Résultat attendu : le français coûte 15 tokens, l'anglais 12.

<details><summary>Indice</summary>

`compter` tient en une ligne : `return len(enc.encode(texte))`. Compare ensuite `tokens_fr` et `tokens_en`.

</details>

In [ ]:
# À toi
fr = "J'adore jouer aux jeux vidéo avec mes amis le week-end."
en = "I love playing video games with my friends on the weekend."

def compter(texte):
    return None

tokens_fr = None
tokens_en = None
langue_la_plus_chere = None     # "fr" ou "en"

print("français :", tokens_fr, "| anglais :", tokens_en, "| la plus chère :", langue_la_plus_chere)

In [ ]:
verifier("Exercice 3 · compter", lambda: compter(fr) == 15 and compter(en) == 12)
verifier("Exercice 3 · tokens_fr / tokens_en", tokens_fr == 15 and tokens_en == 12)
verifier("Exercice 3 · langue_la_plus_chere", langue_la_plus_chere == "fr")

<details><summary>Solution</summary>

```python
fr = "J'adore jouer aux jeux vidéo avec mes amis le week-end."
en = "I love playing video games with my friends on the weekend."

def compter(texte):
    return len(enc.encode(texte))

tokens_fr = compter(fr)
tokens_en = compter(en)
langue_la_plus_chere = "fr" if tokens_fr > tokens_en else "en"
# Le modèle a lu beaucoup plus d'anglais : ses « grosses pièces de Lego » sont des mots anglais.
# Un mot français est souvent coupé en 2 ou 3 morceaux, donc la même idée coûte plus cher en français.
print("français :", tokens_fr, "| anglais :", tokens_en, "| la plus chère :", langue_la_plus_chere)
```

</details>

## Exercice 4 ⭐ · Mots rares, mots fréquents

Un mot fréquent tient dans un seul token, un mot rare est découpé en plusieurs. Pour chaque mot de la liste, compte ses tokens dans le dictionnaire `nb_morceaux` (mot → nombre), puis trouve `mot_le_plus_decoupe` et la liste `mots_en_un_token` (les mots qui tiennent en un seul token).

Résultat attendu : `anticonstitutionnellement` est découpé en 5 morceaux ; `chat` et `computer` tiennent en un token.

<details><summary>Indice</summary>

Une compréhension de dictionnaire : `{mot: compter(mot) for mot in mots}`. Pour le maximum : `max(nb_morceaux, key=nb_morceaux.get)`.

</details>

In [ ]:
# À toi
mots = ["chat", "computer", "ordinateur", "Pokémon", "anticonstitutionnellement", "strawberry"]

nb_morceaux = None            # dictionnaire mot → nombre de tokens
mot_le_plus_decoupe = None
mots_en_un_token = None       # liste

print(nb_morceaux)
print("Le plus découpé :", mot_le_plus_decoupe, "| en un token :", mots_en_un_token)

In [ ]:
verifier("Exercice 4 · nb_morceaux", lambda: nb_morceaux["anticonstitutionnellement"] == 5 and nb_morceaux["ordinateur"] == 2)
verifier("Exercice 4 · mot_le_plus_decoupe", mot_le_plus_decoupe == "anticonstitutionnellement")
verifier("Exercice 4 · mots_en_un_token", lambda: sorted(mots_en_un_token) == ["chat", "computer"])

<details><summary>Solution</summary>

```python
mots = ["chat", "computer", "ordinateur", "Pokémon", "anticonstitutionnellement", "strawberry"]
nb_morceaux = {mot: len(enc.encode(mot)) for mot in mots}
mot_le_plus_decoupe = max(nb_morceaux, key=nb_morceaux.get)
mots_en_un_token = [mot for mot, n in nb_morceaux.items() if n == 1]
print(nb_morceaux)
print("Le plus découpé :", mot_le_plus_decoupe, "| en un token :", mots_en_un_token)
# Pour voir les morceaux : [enc.decode([t]) for t in enc.encode("anticonstitutionnellement")]
```

</details>

## Exercice 5 ⭐⭐ · La table de fréquences du bigramme

Notre mini-modèle de langage, c'est un tableau : « après tel mot, quel mot vient, et combien de fois ? ». Écris `entrainer(liste_phrases)` qui renvoie ce tableau sous la forme `suivants[mot][mot_suivant] = nombre de fois`, en ajoutant `"<début>"` avant chaque phrase et `"<fin>"` après.

Résultat attendu : après « le », le modèle a vu « chat » 5 fois ; après `<début>`, « le » 20 fois.

<details><summary>Indice</summary>

`suivants = defaultdict(Counter)`, puis pour chaque phrase : `mots = ["<début>"] + p.split() + ["<fin>"]` et une boucle sur `zip(mots, mots[1:])` qui fait `suivants[mot][suivant] += 1`.

</details>

In [ ]:
# À toi
def entrainer(liste_phrases):
    suivants = defaultdict(Counter)
    # ... pour chaque phrase, compte les paires (mot, mot suivant)
    return suivants

suivants = entrainer(phrases)
print("Après « le » :", suivants["le"].most_common(5))
print("Après « chat » :", suivants["chat"].most_common())

In [ ]:
verifier("Exercice 5 · après « le »", lambda: suivants["le"]["chat"] == 5 and sum(suivants["le"].values()) == 32)
verifier("Exercice 5 · <début> et <fin>", lambda: suivants["<début>"]["le"] == 20 and suivants["canapé"]["<fin>"] == 1)
verifier("Exercice 5 · taille du vocabulaire", lambda: len(suivants) == 83)

<details><summary>Solution</summary>

```python
def entrainer(liste_phrases):
    suivants = defaultdict(Counter)
    for p in liste_phrases:
        mots = ["<début>"] + p.split() + ["<fin>"]
        for mot, suivant in zip(mots, mots[1:]):
            suivants[mot][suivant] += 1
    return suivants

suivants = entrainer(phrases)
print("Après « le » :", suivants["le"].most_common(5))    # [('chat', 5), ('chien', 5), ('prof', 4), ...]
print("Après « chat » :", suivants["chat"].most_common())
```

</details>

## Exercice 6 ⭐⭐ · Des comptages aux probabilités

Un vrai modèle ne donne pas des comptages mais des **probabilités**. Écris `proba_suivant(mot)` qui renvoie un dictionnaire `mot_suivant → probabilité` (le comptage divisé par le total des suites de ce mot). Pour un mot jamais vu, renvoie un dictionnaire vide.

Résultat attendu : `proba_suivant("le")["chat"]` = 5 / 32 ≈ 0,156, et la somme des probabilités d'un mot vaut 1.

<details><summary>Indice</summary>

`total = sum(suivants[mot].values())`, puis `{m: n / total for m, n in suivants[mot].items()}`. Attention à ne pas diviser par zéro quand le mot est inconnu.

</details>

In [ ]:
# À toi
def proba_suivant(mot):
    return None

p_chat_apres_le = None
print("P(chat | le) =", p_chat_apres_le)
print(proba_suivant("chat"))

In [ ]:
verifier("Exercice 6 · P(chat | le)", lambda: abs(p_chat_apres_le - 5 / 32) < 1e-9)
verifier("Exercice 6 · la somme vaut 1", lambda: abs(sum(proba_suivant("le").values()) - 1) < 1e-9 and proba_suivant("chat")["dort"] == 0.2)
verifier("Exercice 6 · mot inconnu", lambda: proba_suivant("licorne") == {})

<details><summary>Solution</summary>

```python
def proba_suivant(mot):
    if mot not in suivants:
        return {}
    total = sum(suivants[mot].values())
    return {m: n / total for m, n in suivants[mot].items()}

p_chat_apres_le = proba_suivant("le")["chat"]
print("P(chat | le) =", p_chat_apres_le)     # 0.15625
print(proba_suivant("chat"))                  # 5 suites possibles, 0.2 chacune
```

</details>

## Exercice 7 ⭐⭐ · Générer avec une température

Écris `mot_suivant_temp(mot, temperature)` : à température 0, renvoie le mot le plus fréquent ; sinon tire au sort parmi les suites avec `random.choices`, en donnant à chaque candidat le poids `compte ** (1 / temperature)`. Si le mot n'a aucune suite, renvoie `"<fin>"`. Écris ensuite `generer_temp(debut, temperature, max_mots=15)` qui enchaîne les mots jusqu'à `<fin>`.

Résultat attendu : à température 0, après `<début>` vient toujours « le » ; à température 2, en 200 tirages après « le », on voit au moins 5 mots différents.

<details><summary>Indice</summary>

Récupère `mots = list(candidats)` et `comptes = [candidats[m] for m in mots]`. Température 0 : `mots[comptes.index(max(comptes))]`. Sinon : `random.choices(mots, weights=poids)[0]`.

</details>

In [ ]:
# À toi
def mot_suivant_temp(mot, temperature=1.0):
    candidats = suivants[mot]
    return "<fin>"

def generer_temp(debut="<début>", temperature=1.0, max_mots=15):
    texte, mot = [], debut
    # ... boucle : mot = mot_suivant_temp(mot, temperature) ; stop sur "<fin>"
    return " ".join(texte)

for t in [0, 0.7, 2.0]:
    print(f"température {t} :", generer_temp("le", t))

In [ ]:
verifier("Exercice 7 · température 0", lambda: mot_suivant_temp("<début>", 0) == "le" and mot_suivant_temp("licorne", 0) == "<fin>")
random.seed(0)
tirages = {mot_suivant_temp("le", 2.0) for _ in range(200)}
verifier("Exercice 7 · température 2 = variété", lambda: len(tirages) >= 5 and all(m in suivants["le"] for m in tirages))
verifier("Exercice 7 · generer_temp", lambda: generer_temp("je", 0).split()[0] == "joue" and 0 < len(generer_temp("le", 0.7).split()) <= 15)

<details><summary>Solution</summary>

```python
def mot_suivant_temp(mot, temperature=1.0):
    candidats = suivants[mot]
    if not candidats:
        return "<fin>"
    mots = list(candidats)
    comptes = [candidats[m] for m in mots]
    if temperature == 0:
        return mots[comptes.index(max(comptes))]
    poids = [c ** (1 / temperature) for c in comptes]   # température haute = poids aplatis
    return random.choices(mots, weights=poids)[0]

def generer_temp(debut="<début>", temperature=1.0, max_mots=15):
    texte, mot = [], debut
    for _ in range(max_mots):
        mot = mot_suivant_temp(mot, temperature)
        if mot == "<fin>":
            break
        texte.append(mot)
    return " ".join(texte)

for t in [0, 0.7, 2.0]:
    print(f"température {t} :", generer_temp("le", t))
# À 0 : toujours la même phrase (et souvent une boucle). À 2 : presque n'importe quoi.
```

</details>

## Exercice 8 ⭐⭐ · Repérer les hallucinations

Voici 6 réponses données par un petit modèle. Pour chacune, décide si c'est une **hallucination** (`True` : réponse inventée ou fausse) ou une bonne réponse (`False`). Pour les questions de calcul et de lettres, fais vérifier par Python plutôt que de compter de tête !

Résultat attendu : `verdicts` est une liste de 6 booléens, et `nb_hallucinations` leur total.

<details><summary>Indice</summary>

Un événement futur, une personne ou un lieu qui n'existe pas → invention. Pour « combien de r », `"strawberry".count("r")`. Pour la multiplication, `4817 * 2953`.

</details>

In [ ]:
# À toi
reponses_du_modele = [
    {"question": "Qui a gagné la Coupe du monde de football 2031 ?",
     "reponse": "La Coupe du monde 2031 a été remportée par le Brésil, qui a battu l'Allemagne 2-1 en finale."},
    {"question": "Combien font 12 × 12 ?",
     "reponse": "12 × 12 = 144."},
    {"question": "Qui est le maire de Trifouillis-les-Oies ?",
     "reponse": "Le maire actuel de Trifouillis-les-Oies est Jean-Pierre Martin, élu en 2020 avec 54 % des voix."},
    {"question": "Combien de lettres r dans le mot strawberry ?",
     "reponse": "Le mot strawberry contient 2 lettres r."},
    {"question": "Quelle est la capitale de la France ?",
     "reponse": "La capitale de la France est Paris."},
    {"question": "Combien font 4 817 × 2 953 ?",
     "reponse": "Le résultat de cette multiplication est 8 452 917."},
]

verdicts = [None, None, None, None, None, None]     # True = hallucination, False = bonne réponse
nb_hallucinations = None

for r, v in zip(reponses_du_modele, verdicts):
    print("HALLUCINATION" if v else "ok           ", "|", r["question"])

In [ ]:
verifier("Exercice 8 · verdicts", verdicts == [True, False, True, True, False, True])
verifier("Exercice 8 · nb_hallucinations", nb_hallucinations == 4)

<details><summary>Solution</summary>

```python
reponses_du_modele = [
    {"question": "Qui a gagné la Coupe du monde de football 2031 ?",
     "reponse": "La Coupe du monde 2031 a été remportée par le Brésil, qui a battu l'Allemagne 2-1 en finale."},
    {"question": "Combien font 12 × 12 ?", "reponse": "12 × 12 = 144."},
    {"question": "Qui est le maire de Trifouillis-les-Oies ?",
     "reponse": "Le maire actuel de Trifouillis-les-Oies est Jean-Pierre Martin, élu en 2020 avec 54 % des voix."},
    {"question": "Combien de lettres r dans le mot strawberry ?", "reponse": "Le mot strawberry contient 2 lettres r."},
    {"question": "Quelle est la capitale de la France ?", "reponse": "La capitale de la France est Paris."},
    {"question": "Combien font 4 817 × 2 953 ?", "reponse": "Le résultat de cette multiplication est 8 452 917."},
]
print("r dans strawberry :", "strawberry".count("r"))   # 3, pas 2
print("4817 × 2953 =", 4817 * 2953)                     # 14 224 601, pas 8 452 917
verdicts = [True,    # 2031 n'a pas eu lieu : inventé
            False,   # 144 est juste
            True,    # ce village n'existe pas : nom inventé avec assurance
            True,    # il y a 3 r (le modèle voit des tokens, pas des lettres)
            False,   # Paris
            True]    # faux calcul dit avec assurance
nb_hallucinations = sum(verdicts)
for r, v in zip(reponses_du_modele, verdicts):
    print("HALLUCINATION" if v else "ok           ", "|", r["question"])
```

</details>

## Exercice 9 ⭐⭐ · Python comme juge

Pose au modèle la question « Combien de lettres e dans le mot anticonstitutionnellement ? » avec `demander`, puis **extrais le nombre** de sa réponse avec une expression régulière (`re.search(r"\d+", ...)`), et compare avec ce que Python compte vraiment.

Résultat attendu : `nombre_python` = 3, `nombre_modele` = le nombre extrait (un entier), `le_modele_a_raison` = `True` ou `False`. (En mode démo, le modèle répond 2 : il a tort.)

<details><summary>Indice</summary>

`trouve = re.search(r"\d+", reponse_modele)` puis `int(trouve.group(0))` si `trouve` n'est pas `None`. Pour Python : `mot.count("e")`.

</details>

In [ ]:
# À toi
mot = "anticonstitutionnellement"
question = f"Combien de lettres e dans le mot {mot} ?"

reponse_modele = None      # la réponse (texte) du modèle
nombre_modele = None       # le nombre extrait de la réponse
nombre_python = None       # ce que Python compte
le_modele_a_raison = None

print("Modèle :", reponse_modele)
print("Modèle dit", nombre_modele, "| Python dit", nombre_python, "| le modèle a raison :", le_modele_a_raison)

In [ ]:
verifier("Exercice 9 · nombre_python", nombre_python == 3)
verifier("Exercice 9 · nombre_modele extrait", isinstance(nombre_modele, int))
verifier("Exercice 9 · verdict", lambda: le_modele_a_raison == (nombre_modele == nombre_python))

<details><summary>Solution</summary>

```python
mot = "anticonstitutionnellement"
question = f"Combien de lettres e dans le mot {mot} ?"

reponse_modele = demander(question)
trouve = re.search(r"\d+", reponse_modele)
nombre_modele = int(trouve.group(0)) if trouve else None
nombre_python = mot.count("e")
le_modele_a_raison = nombre_modele == nombre_python

print("Modèle :", reponse_modele)
print("Modèle dit", nombre_modele, "| Python dit", nombre_python, "| le modèle a raison :", le_modele_a_raison)
# Le modèle voit ['ant', 'icon', 'stitution', 'nel', 'lement'] : aucune lettre isolée. Python, lui, voit les lettres.
```

</details>

## Exercice 10 ⭐⭐ · Le sac de mots (la méthode d'avant)

Refais le détecteur de spam de la leçon : un `CountVectorizer` qui compte les mots, et un `MultinomialNB` entraîné sur `textes` / `etiquettes`. Écris `predire_sac(texte)` qui renvoie `"spam"` ou `"pas spam"`, puis classe les 3 nouveaux messages.

Résultat attendu : `["spam", "pas spam", "spam"]`. Le 3e message est un **piège** : il contient « gagné », « gratuit », « voyage »... mais c'est un ami qui fête sa tombola.

<details><summary>Indice</summary>

`sac = CountVectorizer(); X = sac.fit_transform(textes); classifieur = MultinomialNB().fit(X, etiquettes)`. Pour prédire : `classifieur.predict(sac.transform([texte]))[0]` donne 1 ou 0.

</details>

In [ ]:
# À toi
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

sac = None
classifieur = None

def predire_sac(texte):
    return None

nouveaux = [
    "Cliquez vite, cadeau gratuit et urgent",
    "On se voit demain au foot ?",
    "Hier j'ai gagné un voyage gratuit à la tombola du club, on fête ça ?",   # le piège
]
verdicts_sac = None     # liste des 3 verdicts
print(verdicts_sac)

In [ ]:
verifier("Exercice 10 · vocabulaire appris", lambda: len(sac.vocabulary_) == 105)
verifier("Exercice 10 · verdicts_sac", verdicts_sac == ["spam", "pas spam", "spam"])

<details><summary>Solution</summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

sac = CountVectorizer()                       # chaque message devient un comptage de mots
X = sac.fit_transform(textes)
classifieur = MultinomialNB().fit(X, etiquettes)

def predire_sac(texte):
    return "spam" if classifieur.predict(sac.transform([texte]))[0] == 1 else "pas spam"

nouveaux = [
    "Cliquez vite, cadeau gratuit et urgent",
    "On se voit demain au foot ?",
    "Hier j'ai gagné un voyage gratuit à la tombola du club, on fête ça ?",
]
verdicts_sac = [predire_sac(t) for t in nouveaux]
print(verdicts_sac)   # ['spam', 'pas spam', 'spam'] : le sac de mots tombe dans le piège
```

</details>

## Exercice 11 ⭐⭐⭐ · Sac de mots contre LLM

Écris `predire_llm(texte)` qui demande au modèle de classer le message avec le prompt système `"Tu es un filtre anti-spam. Réponds uniquement par : spam ou pas spam."` (`max_new_tokens=5, temperature=0`) et **normalise** sa réponse en `"spam"` ou `"pas spam"`. Compare ensuite les deux méthodes sur `nouveaux` : `desaccords` = la liste des messages où elles ne sont pas d'accord.

Résultat attendu (mode démo) : un seul désaccord, le message piège. Le LLM lit la phrase entière et comprend le contexte. (Avec le vrai petit modèle, le résultat peut varier : note ce que tu observes.)

<details><summary>Indice</summary>

Construis la liste de messages `[{"role": "system", ...}, {"role": "user", "content": texte}]`, puis `"pas spam" if "pas" in reponse.lower() else "spam"`.

</details>

In [ ]:
# À toi
SYSTEME_SPAM = "Tu es un filtre anti-spam. Réponds uniquement par : spam ou pas spam."

def predire_llm(texte):
    return None

desaccords = None

for t in nouveaux:
    print(f"sac : {predire_sac(t)!s:9} | llm : {predire_llm(t)!s:9} | {t}")
print("Désaccords :", desaccords)

In [ ]:
verifier("Exercice 11 · predire_llm normalise", lambda: predire_llm(nouveaux[1]) == "pas spam" and predire_llm(nouveaux[0]) == "spam")
verifier("Exercice 11 · desaccords (mode démo)", lambda: desaccords == [nouveaux[2]])

<details><summary>Solution</summary>

```python
SYSTEME_SPAM = "Tu es un filtre anti-spam. Réponds uniquement par : spam ou pas spam."

def predire_llm(texte):
    reponse = llm([{"role": "system", "content": SYSTEME_SPAM}, {"role": "user", "content": texte}],
                  max_new_tokens=5, temperature=0)
    return "pas spam" if "pas" in reponse.lower() else "spam"

desaccords = [t for t in nouveaux if predire_sac(t) != predire_llm(t)]

for t in nouveaux:
    print(f"sac : {predire_sac(t)!s:9} | llm : {predire_llm(t)!s:9} | {t}")
print("Désaccords :", desaccords)
# Le sac de mots voit « gagné » + « gratuit » et crie au spam ; le modèle de langage lit la phrase entière.
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : un trigramme qui ne tourne plus en rond

Le bigramme regarde **un** mot en arrière et boucle (« le chat dort dans le chat dort... »). Construis un **trigramme** qui regarde **deux** mots en arrière :
1. `entrainer_trigramme(liste_phrases)` renvoie `trigramme[(mot1, mot2)][mot_suivant] = nombre` (commence chaque phrase par deux `"<début>"`, finis par `"<fin>"`) ;
2. `generer_trigramme(mot1, mot2, max_mots=15)` part des deux mots donnés, choisit à chaque fois la suite la plus fréquente, s'arrête sur `<fin>` ou sur un couple jamais vu, et renvoie la phrase **en commençant par `mot1 mot2`** ;
3. `cout_tokens` = le nombre de tokens `tiktoken` de `generer_trigramme("le", "chat")`.

Résultat attendu : `generer_trigramme("le", "chat")` → `"le chat dort sur le canapé"`, et le couple `("<début>", "<début>")` a 8 suites possibles.

<details><summary>Indice</summary>

Même idée que l'exercice 5 avec `zip(mots, mots[1:], mots[2:])` et une clé `(a, b)`. Dans la génération, garde les deux derniers mots : `mot1, mot2 = mot2, suivant`.

</details>

In [ ]:
# À toi
def entrainer_trigramme(liste_phrases):
    trigramme = defaultdict(Counter)
    return trigramme

def generer_trigramme(mot1, mot2, max_mots=15):
    texte = [mot1, mot2]
    return " ".join(texte)

trigramme = entrainer_trigramme(phrases)
cout_tokens = None

print("bigramme  :", generer_temp("le", 0))
print("trigramme :", generer_trigramme("le", "chat"))
print("coût en tokens :", cout_tokens)

In [ ]:
verifier("Exercice 12 · table du trigramme", lambda: trigramme[("le", "chat")]["dort"] == 1 and len(trigramme[("<début>", "<début>")]) == 8)
verifier("Exercice 12 · génération", lambda: generer_trigramme("le", "chat") == "le chat dort sur le canapé" and generer_trigramme("licorne", "rose") == "licorne rose")
verifier("Exercice 12 · cout_tokens", lambda: cout_tokens == len(enc.encode(generer_trigramme("le", "chat"))))

<details><summary>Solution</summary>

```python
def entrainer_trigramme(liste_phrases):
    trigramme = defaultdict(Counter)
    for p in liste_phrases:
        mots = ["<début>", "<début>"] + p.split() + ["<fin>"]
        for a, b, c in zip(mots, mots[1:], mots[2:]):
            trigramme[(a, b)][c] += 1
    return trigramme

def generer_trigramme(mot1, mot2, max_mots=15):
    texte = [mot1, mot2]
    for _ in range(max_mots):
        if not trigramme[(mot1, mot2)]:          # couple jamais vu
            break
        suivant = trigramme[(mot1, mot2)].most_common(1)[0][0]
        if suivant == "<fin>":
            break
        texte.append(suivant)
        mot1, mot2 = mot2, suivant
    return " ".join(texte)

trigramme = entrainer_trigramme(phrases)
cout_tokens = len(enc.encode(generer_trigramme("le", "chat")))

print("bigramme  :", generer_temp("le", 0))              # boucle : le chat dort dans le chat dort...
print("trigramme :", generer_trigramme("le", "chat"))    # le chat dort sur le canapé
print("coût en tokens :", cout_tokens)
# Avec 2 mots de contexte, le modèle « se souvient » mieux. Un vrai LLM regarde des milliers de mots en arrière.
```

</details>

## Bravo !

Tu as manipulé les trois idées de la séance : les **tokens** (le modèle ne voit pas les lettres), la **prédiction du mot suivant** (bigramme, probabilités, température) et les **limites** d'un modèle (hallucinations, calculs, sac de mots sans contexte). Pour aller plus loin, ajoute tes propres phrases à `phrases` et regarde comment ton trigramme change.